# 03 — One-stage anchor-based detector: RetinaNet (torchvision)

RetinaNet ResNet-50-FPN (COCO weights). There is no proposal stage: every anchor on P3–P7 is
classified directly. An anchor is positive if IoU ≥ 0.5 with a GT box, negative if IoU < 0.4.
Without the RPN, almost all of the ~170k anchors per frame are easy background, which would dominate
the loss; the focal loss down-weights these easy examples.

| Run | Change | Why |
|---|---|---|
| `retina_base` | default anchors 32–512 px, ratios 0.5 / 1 / 2, L1 box loss | baseline |
| `retina_anchors` | anchors **from the data**: sizes start at the 5th percentile of object size, ratios = k-means of h/w | default 32 px anchors are too big for most objects |
| `retina_giou` | better of the two + **GIoU** box loss | loss modification |

Approximate time on a T4 with `FAST=True`: ~0.5 h per run, ~1.7 h in total (~4 h without FAST).

## 0. Setup
Works on **Kaggle** (add the competition as input, GPU, Internet on), **Google Colab** (GPU T4,
Kaggle token as Colab secret `KAGGLE_API_TOKEN`, or `kaggle.json` in the root of Google Drive) and **a laptop** (Kaggle token in `<repo>/.kaggle_token`, git-ignored;
data and results go to `~/vdet_data` and `~/vdet_work`; without a GPU use it only with `SMOKE = True`).
On Colab all runs are saved to `MyDrive/vdet`, so after a disconnect just run the notebook again:
finished runs are skipped and interrupted runs continue from the last epoch.

In [ ]:
# ---- Settings ----
SMOKE = False   # True: tiny run on a few images, only to check that the notebook works end to end
FAST = True     # True: every 20th train frame and fewer epochs (~2x faster); False: full protocol
REPO = "https://github.com/LidaSokha03/AI_HW2.git"   # shared code is in src/ of this repository
PRETRAINED = True   # COCO weights from torchvision

In [ ]:
import os
import sys
import subprocess

if "google.colab" in sys.modules:  # Colab: keep runs and results on Google Drive
    from google.colab import drive
    drive.mount("/content/drive")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'pycocotools', 'kaggle>=2.0'], check=True)

# Shared code (src/vdet.py, src/tvdet.py): local copy if present, otherwise cloned from GitHub
CANDIDATES = ["src", "../src", ".", "notebooks"]
SRC = os.environ.get("VDET_SRC") or next((p for p in CANDIDATES if os.path.exists(f"{p}/vdet.py")), None)
if SRC is None:
    if os.path.exists("/tmp/AI_HW2"):
        subprocess.run(["git", "-C", "/tmp/AI_HW2", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "/tmp/AI_HW2"], check=True)
    SRC = next(f"/tmp/AI_HW2/{p}" for p in ["src", "notebooks"] if os.path.exists(f"/tmp/AI_HW2/{p}/vdet.py"))
sys.path.insert(0, os.path.abspath(SRC))

import vdet

# Same frame subset for every model (see the protocol below). First run: ~5 min to prepare.
TRAIN_STEP = 20 if FAST else 10
S = vdet.setup(train_step=TRAIN_STEP, val_step=5, max_side=1600, smoke=SMOKE)
print({k: S[k] for k in ["platform", "work", "prep"]})

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tvdet

NB = "03_retinanet"
GT = f"{S['prep']}/val_gt.json"
VAL = vdet.val_items(S)[:20] if SMOKE else vdet.val_items(S)
TEST = vdet.test_items(S)[:30] if SMOKE else vdet.test_items(S)
COLS = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR500", "ms_per_img", "FPS"]
print("device:", tvdet.DEVICE)

### Common protocol (identical in all model notebooks)
| Item | Value |
|---|---|
| Split | given train / val split, which is already **by video** (56 / 7 videos, no leakage) |
| Frames | every 20th train frame (≈1,200, `FAST=True`) or every 10th (≈2,400) and every 5th val frame (≈570): neighbouring frames are near-duplicates |
| Resolution | frames stored with longer side ≤ 1600 px; model input longer side **1280 px** unless the experiment changes it |
| Targets | `iscrowd=1` boxes (ignored regions, "others") are removed from training targets |
| Metric | COCO mAP@[.5:.95], AP50, AP_small… with `pycocotools` on val at **original resolution**, ignored regions as `iscrowd`, ≤ 500 detections per frame (same as the leaderboard) |
| Speed | ms per frame on the same GPU, batch 1, including image reading and NMS |
| Budget | ≈ 0.5 GPU-hour (T4) per run with `FAST=True`, ≈ 1 hour without |
| Seed | 42 |

### Training settings
Same as Faster R-CNN, except lr 0.0025 (torchvision reference 0.01 for batch 16, scaled linearly).
Default focal loss α = 0.25, γ = 2.

## 1. Anchors grounded in the data
Object sizes are measured after resizing the frame to the model input (longer side 1280 px).
"Coverage" = share of GT boxes whose best anchor shape reaches IoU ≥ 0.5 (or 0.4) when centred
on the object (an upper bound: in reality the anchor centre is on the stride grid).

In [ ]:
def anchor_shapes(sizes, ratios):
    """(w, h) of all anchors; torchvision: w = s / sqrt(r), h = s * sqrt(r), r = h / w."""
    return np.array([(s / np.sqrt(r), s * np.sqrt(r)) for lvl, rs in zip(sizes, ratios) for s in lvl for r in rs])


def anchor_coverage(gt_json, sizes, ratios, model_side=1280):
    import json
    gt = json.load(open(gt_json))
    ims = {im["id"]: im for im in gt["images"]}
    wh = np.array([(a["bbox"][2], a["bbox"][3], model_side / max(ims[a["image_id"]]["width"], ims[a["image_id"]]["height"]))
                   for a in gt["annotations"] if a["iscrowd"] == 0])
    w, h = wh[:, 0] * wh[:, 2], wh[:, 1] * wh[:, 2]
    A = anchor_shapes(sizes, ratios)
    inter = np.minimum(w[:, None], A[None, :, 0]) * np.minimum(h[:, None], A[None, :, 1])
    best = (inter / (w[:, None] * h[:, None] + A[None, :, 0] * A[None, :, 1] - inter)).max(1)
    size = np.where(w * h < 32 ** 2, "small", np.where(w * h < 96 ** 2, "medium", "large"))
    df = pd.DataFrame({"best_iou": best, "size": size})
    out = {"IoU>=0.5": (df.best_iou >= 0.5).mean(), "IoU>=0.4": (df.best_iou >= 0.4).mean()}
    out.update({f"IoU>=0.5 ({s})": (df[df["size"] == s].best_iou >= 0.5).mean() for s in ["small", "medium", "large"]})
    return pd.Series(out).round(3)


default_sizes, default_ratios = tvdet.default_retinanet_anchors()
data_sizes, data_ratios, size_stats = tvdet.suggest_anchors(f"{S['prep']}/train_gt.json")
print("object size sqrt(w*h) at 1280 px, percentiles:", size_stats.to_dict())
print("default anchors:", default_sizes, default_ratios[0])
print("data anchors:   ", data_sizes, data_ratios[0])
display(pd.DataFrame({"default": anchor_coverage(GT, default_sizes, default_ratios),
                      "from data": anchor_coverage(GT, data_sizes, data_ratios)}))

# number of anchors per 1280x720 frame (P3-P7, strides 8..128)
n_loc = sum(int(np.ceil(720 / s)) * int(np.ceil(1280 / s)) for s in [8, 16, 32, 64, 128])
print(f"anchors per frame: {n_loc * 9:,} (vs. 1000-2000 proposals in Faster R-CNN)")

In [ ]:
EPOCHS = 1 if SMOKE else (4 if FAST else 6)
TRAIN_KW = dict(epochs=EPOCHS, batch_size=4, lr=0.0025,
                max_iters_per_epoch=3 if SMOKE else None, eval_limit=20 if SMOKE else None)
CONFIGS = {
    "retina_base": dict(),
    "retina_anchors": dict(anchor_sizes=data_sizes, aspect_ratios=data_ratios),
}
results, val_preds, histories = {}, {}, {}


def run_retina(name, cfg, reg_loss="l1"):
    """Train (or resume / reuse) one run, evaluate the best epoch on val and log the result."""
    model = tvdet.build_retinanet(pretrained=PRETRAINED, reg_loss=reg_loss, **cfg)
    histories[name] = tvdet.train(model, S, name, **TRAIN_KW)
    model = tvdet.load_best(model, S, name)
    preds = tvdet.predict(model, VAL)
    metrics = vdet.coco_eval(GT, preds, per_class=True)
    metrics.update(tvdet.speed(model, VAL, n=5 if SMOKE else 100))
    vdet.log_result(S, NB, name, metrics, epochs=EPOCHS, reg_loss=reg_loss,
                    anchors="data" if "anchor_sizes" in cfg else "default")
    results[name], val_preds[name] = metrics, preds
    return model, pd.DataFrame([metrics], index=[name])[COLS]

## 2. Baseline (default anchors)

In [ ]:
model, row = run_retina("retina_base", CONFIGS["retina_base"])
row

## 3. Experiment 1: anchors from the data

In [ ]:
model, row = run_retina("retina_anchors", CONFIGS["retina_anchors"])
row

## 4. Experiment 2: GIoU box loss (on top of the better anchors)

In [ ]:
base_cfg = max(["retina_base", "retina_anchors"], key=lambda n: results[n]["mAP"])
print("GIoU run uses the anchors of:", base_cfg)
CONFIGS["retina_giou"] = CONFIGS[base_cfg]
model, row = run_retina("retina_giou", CONFIGS["retina_giou"], reg_loss="giou")
row

## 5. Comparison (and with the tuned Faster R-CNN)
Faster R-CNN results are read from `results/02_faster_rcnn.csv` (this work dir, Kaggle inputs or the
repository's `results/` folder).

In [ ]:
table = vdet.results_table(results)
display(table[COLS])
vdet.plot_size_bars(table, "RetinaNet: AP by object size", save=f"{S['work']}/figs/retina_ap_by_size.png")
print("AP per class:")
display(table[[f"AP_{c}" for c in vdet.CLASS_NAMES]].astype(float).round(3))

fig, ax = plt.subplots(figsize=(7, 4))
for name, h in histories.items():
    ax.plot(h["epoch"], h["mAP"], marker="o", label=name)
ax.set_xlabel("epoch"); ax.set_ylabel("val mAP"); ax.set_title("RetinaNet: val mAP per epoch"); ax.legend()
plt.tight_layout(); plt.savefig(f"{S['work']}/figs/retina_curves.png", dpi=110); plt.show()

allres = vdet.load_results(S)
if len(allres):
    others = allres[allres.notebook == "02_faster_rcnn"].set_index("run")
    display(pd.concat([others, table]).loc[:, COLS].astype(float).round(4))
else:
    print("No Faster R-CNN results found yet (run 02 or put results/02_faster_rcnn.csv into the repository).")

## 6. Errors of the best run

In [ ]:
best = table["mAP"].astype(float).idxmax()
print("best run:", best)
miss = vdet.error_breakdown(S, val_preds[best])
ids = vdet.worst_frames(S, val_preds[best], k=2)
vdet.plot_errors(S, val_preds[best], ids, save=f"{S['work']}/figs/retina_errors.png")

## 7. Submission with the best run

In [ ]:
best_model = tvdet.load_best(tvdet.build_retinanet(pretrained=False, **CONFIGS[best]), S, best)
vdet.write_submission(S, tvdet.predict(best_model, TEST), best)